# Module 8: When Parallel Trends Fails

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

[Module 7](Module_07_Testing_Parallel_Trends.ipynb) found that Summit County
was falling at 12 percent a year before the training existed, against 4 to 6
for everyone else.

Four responses are available. This module runs all four against a known
answer, and the result is not the one most people expect: **the simple fix
works and the sophisticated one destroys the estimate.**

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

f["lo"] = np.log(f["n_arrests"])
pi = pd.PeriodIndex(f["year_month"], freq="M")
f["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
f["settled"] = ((f["trained"] == 1) & (f["period"] == "after")).astype(float)
f["phase"] = ((f["trained"] == 1) & (f["period"] == "phase")).astype(float)

pct = lambda b: 100 * (np.exp(b) - 1)

BASE = "n_uof ~ C(agency_id) + C(year_month) + settled + phase"
TRENDS = "n_uof ~ C(agency_id) + C(year_month) + C(agency_id):yr + settled + phase"


def run(formula, data):
    z = smf.glm(formula, data, family=sm.families.Poisson(),
                offset=data["lo"]).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi)

## 2. The four responses

| Response | The idea |
|---|---|
| **Do nothing** | report it and note the violation |
| **Drop the agency** | it does not meet the design's requirement |
| **Agency specific trends** | let every agency have its own slope and compare deviations |
| **Both** | belt and braces |

In [ ]:
rows = []
for label, formula, data in [
        ("do nothing, keep Summit County", BASE, f),
        ("drop Summit County", BASE, f[f["agency_id"] != "A007"]),
        ("agency specific linear trends", TRENDS, f),
        ("both", TRENDS, f[f["agency_id"] != "A007"])]:
    e, lo, hi = run(formula, data)
    rows.append({"response": label, "estimate": f"{e:+.1f}%",
                 "95 percent interval": f"[{lo:+.1f}, {hi:+.1f}]",
                 "width": round(hi - lo, 1),
                 "covers zero": "yes" if lo < 0 < hi else ""})
rows.append({"response": "THE TRUTH", "estimate": f"{TRUTH:+.1f}%",
             "95 percent interval": "", "width": "", "covers zero": ""})
pd.DataFrame(rows).set_index("response")

**Dropping the agency recovers the truth.** Everything else does worse.

Doing nothing gives 17.0 percent, which is the violation showing up as an
inflated effect, exactly as expected.

The two specifications with agency specific trends give 8.3 and 7.7 percent,
and **their intervals now include zero.** A correctly estimated 12 percent
effect has been turned into a null result by the fix.

## 3. Why the sophisticated fix fails

The problem is collinearity. The program starts in late 2023, near the end of
the series, so a downward step at that point and a steeper downward slope over
the whole period explain the same data.

In [ ]:
g = f[f["agency_id"] == "A001"].copy()
print(f"  months before the program:  {(g['period'] == 'before').sum()}")
print(f"  months after it settled:    {(g['period'] == 'after').sum()}")
print(f"  correlation between the settled indicator and time: "
      f"{np.corrcoef(g['yr'], (g['period'] == 'after').astype(float))[0, 1]:.3f}")

With the treated period sitting in the last third of the series, an agency
specific trend term and a treatment indicator are fighting over the same
variation. The trend absorbs part of the effect, the estimate shrinks, and
the standard error nearly doubles because the two terms cannot be separated.

This is the same mechanism as Time Series Advanced
[Module 11](../../../Time_Series/Advanced/Module_11_Interrupted_Time_Series.md),
where adding a slope term to a pure level change produced a significant slope
that did not exist.

**Agency specific trends are not a general remedy for a failed parallel trends
test.** They work when the treated period is long relative to the pre period
and the violation is genuinely a smooth slope difference. Neither holds here.

## 4. What to do instead

| Situation | Response |
|---|---|
| One agency, identifiable, with a documented reason | drop it, say so, report both estimates |
| Several agencies differ, no obvious reason | the design is not usable; say so |
| The violation is small and the treated period is long | agency specific trends are worth trying |
| The comparison group can be rebuilt | restrict it to agencies whose trends match |
| Nothing works | report the pre trend difference and let the reader discount |

The fourth row is worth a demonstration, because it is the option most often
forgotten.

In [ ]:
keep = [a for a in TRAINED if a != "A007"]
pre = f[f["period"] == "before"]

matched = []
for a in COMPARISON:
    s = pre[pre["agency_id"] == a]
    z = smf.glm("n_uof ~ yr", s, family=sm.families.Poisson(),
                offset=s["lo"]).fit()
    if -8 < pct(z.params["yr"]) < -2:              # close to the treated group's 5%
        matched.append(a)

print(f"  comparison agencies whose own pre trend is between 2 and 8 percent down:")
print(f"    {', '.join(NAME[a].split()[0] for a in matched)}\n")
e, lo, hi = run(BASE, f[f["agency_id"].isin(keep + matched)])
print(f"  estimate from the trend matched comparison group: "
      f"{e:+.1f}%  [{lo:+.1f}, {hi:+.1f}]")
e, lo, hi = run(BASE, f[f["agency_id"] != "A007"])
print(f"  estimate from all seven comparison agencies:      "
      f"{e:+.1f}%  [{lo:+.1f}, {hi:+.1f}]")
print(f"  the truth:                                        {TRUTH:+.1f}%")

Restricting the comparison group to agencies on similar trends gives
essentially the same answer, 12.3 against 12.6 percent, and an interval of
almost exactly the same width.

**The interval did not widen, and there is a reason.** Three agencies were
dropped but all three were small: the four that remain still include Ashfell,
which carries roughly three quarters of the comparison group's incidents.
Dropping agencies costs precision in proportion to the incidents they
contributed, not in proportion to how many of them there were, which is the
lesson from [Module 4](Module_04_Building_A_Comparison_Group.ipynb) arriving
from a different direction.

Matching on trends is worth doing when it changes the answer or when a
reviewer will ask. Here it does neither, and reporting that is the point.

## 5. What to write

> *One treated agency was excluded because its use of force rate was already
> declining at 12.0 percent a year before the program began, against 4 to 6
> percent at every other agency in the study, with non overlapping intervals.
> The agency began an internal reform in 2019. Including it raises the
> estimate from 12.6 to 17.0 percent. A specification with agency specific
> linear trends was also fitted and is not reported as the primary result: the
> treated period occupies the final third of the series, so the trend terms
> are nearly collinear with the treatment indicator and the interval widens to
> include zero.*

Every sentence there is checkable, and the specification that was tried and
set aside is disclosed rather than buried.

## Exercise

Summit County was dropped from the **treated** group. Find out what happens if
it is moved to the comparison group instead, which is a mistake that looks
reasonable.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    moved = f.copy()
    moved["settled"] = ((moved["agency_id"].isin(
        [a for a in TRAINED if a != "A007"])) & (moved["period"] == "after")).astype(float)
    moved["phase"] = ((moved["agency_id"].isin(
        [a for a in TRAINED if a != "A007"])) & (moved["period"] == "phase")).astype(float)
    rows = []
    for label, data in [("dropped entirely", f[f["agency_id"] != "A007"]),
                        ("moved to the comparison group", moved)]:
        e, lo, hi = run(BASE, data)
        rows.append({"what was done with Summit County": label,
                     "estimate": f"{e:+.1f}%",
                     "95 percent interval": f"[{lo:+.1f}, {hi:+.1f}]"})
    rows.append({"what was done with Summit County": "THE TRUTH",
                 "estimate": f"{TRUTH:+.1f}%", "95 percent interval": ""})
    display(pd.DataFrame(rows).set_index("what was done with Summit County"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Moving Summit County into the comparison group makes the estimate **smaller**,
and it is wrong in a new way.

Summit County took the training. Putting it among the untrained agencies means
the comparison group contains a treated unit, so the comparison group's
decline now includes a real program effect, and dividing by it removes part of
what the study is measuring. This is contamination, deliberately introduced,
and [Module 13](Module_13_Spillover_And_Contamination.ipynb) is about it.

It also still carries its own steep pre trend, now pushing the comparison
group's decline further down for a second unrelated reason.

**An agency that fails the parallel trends test is not thereby a control.**
The only defensible moves are to exclude it from the study or to report it
separately. Reassigning it is neither.

</details>

---

**Next:** [Module 9: Regression to the Mean](Module_09_Regression_To_The_Mean.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*